In [14]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time 
import traceback

class SparkConfig :
    def creat_sparksession() :
        spark = SparkSession.builder.config("spark.driver.memory" , "8g") \
                            .appName("clean data") \
                            .master("local[*]") \
                            .getOrCreate()
        spark.sparkContext._jsc.hadoopConfiguration().set("fs.defaultFS", "file:///")
        return spark

In [2]:
spark = SparkConfig.creat_sparksession()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 22:28:51 WARN Utils: Your hostname, Admin-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 22:28:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 22:28:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
class DataLoader :
    def __init__(self) :
        self.spark = SparkConfig.create_sparksession() 
    def read_works() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/works/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            works = self.spark.read.format("parquet").load(file) 
            
            works = works.withColumn("cover_id" , col("cover_id").cast("long"))
            for c in works.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in works.columns :
                    works = works.withColumn(c , lit(None).cast("string"))
            list_file.append(works)
        works = reduce(lambda a,b : a.union(b) , list_file)
        works.coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/works/")
        return works

    def read_editions() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/editions/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            editions = self.spark.read.format("parquet").load(file) 
            if "number_of_pages" in editions.columns :
                editions = editions.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            for c in editions.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in editions.columns :
                    editions = editions.withColumn(c , lit(None).cast("string"))
            list_file.append(editions)
        editions = reduce(lambda a,b : a.union(b) , list_file)
        editions.coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/editions/")
        return editions 
    
    def read_authors() :        
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/authors/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        for file in files :
            authors = self.spark.read.format("parquet").load(file) 
            if "number_of_pages" in authors.columns :
                authors = authors.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            list_file.append(authors)
        authors = reduce(lambda a,b : a.unionByName(b , allowMissingColumns = True) , list_file)
        authors.coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/authors/")
        return authors

In [4]:
spark

In [5]:
works = spark.read.parquet("data/works.parquet")

In [17]:
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent

works_path = BASE_DIR / "clean_data" / "data" / "works.parquet"

print(works_path)

NameError: name '__file__' is not defined

In [6]:
editions = spark.read.parquet("data/editions.parquet")

In [7]:
authors = spark.read.parquet("data/authors.parquet")

In [8]:
class DataClean :
    def work_clean(works) :
        print("\n" + "="*60 )
        print("====== Starting Clean works ======")
        print("="*60 )
        # Chọn các cột cần để phân tích
        works = works.select("key", "title", "first_publish_year", "edition_count", "subject", "authors" )
        print("====== Missing Value in works ======")
        works.select([count(when (col(c).isNull() , c)).alias(c) for c in works.columns ]).show()
        # Làm sạch cột key và first_pubnlish_year 
        work_key = works.withColumn("key" , regexp_extract(col("key") , r"[A-Z]+\d+[A-Z]", 0)) \
                        .withColumnRenamed("key", "work_id") \
                        .withColumn("first_publish_year" , col("first_publish_year").cast("int")) # chuyển đổi kiểu dữ liệu cột first_publish_year
        # Xử lý giá trị missing value bằng cách lấy trung vị
        median = work_key.approxQuantile("first_publish_year", [0.5], 0.01)[0]
        work_key = work_key.fillna({"first_publish_year" : int(median)})
        # Explode cột subject 
        work_explode_sub = work_key.withColumn("subject" , regexp_replace("subject" , r"[\]\[]", "")) \
                                .withColumn("subject", explode(split("subject" , ",")))
        # Làm sạch cột subject
        work_explode_sub = work_explode_sub.withColumn("subject", regexp_replace("subject", r'form:|genre:|"', ""))
        # Làm sạch bảng work_explode_sub
        work_explode_sub = work_explode_sub.filter(col("subject").isNotNull() & ~col("subject").rlike("\\\\|[0-9]"))
        # Xóa bỏ khoảng trắng, lọc các dữ liệu sạch
        work_explode_sub = work_explode_sub.orderBy("subject") \
                .withColumn("subject", trim(col("subject"))) \
                .filter(~col("subject").rlike("&|.com$|@|^:|^\\.") & trim(col("subject") != "") ) \
                .withColumn("subject", regexp_replace("subject" ,"'|\\*|-" , "")) 
        # Làm sạch bảng work_explode_sub
        work_explode_sub = work_explode_sub.filter(col("subject").isNotNull() & ~col("subject").rlike("\\\\|[0-9]"))
        # Xóa bỏ khoảng trắng, lọc các dữ liệu sạch
        work_explode_sub = work_explode_sub.orderBy("subject") \
                .withColumn("subject", trim(col("subject"))) \
                .filter(~col("subject").rlike("&|.com$|@|^:|^\\.") & trim(col("subject") != "") ) \
                .withColumn("subject", regexp_replace("subject" ,"'|\\*|-" , "")) 
        work_explode_sub.show(5)
        print("====== Clean works sucessfully ======")
        return work_explode_sub

    def edition_clean(editions) :
        print("\n" + "="*60 )
        print("====== Starting Clean editions ======")
        print("="*60 )
        editions = editions.select("key", "works", "title", "publish_date", "publishers", "languages", "number_of_pages", "isbn_13")
        # Tạo work schema
        work_schema = ArrayType(
            StructType([
                StructField("key", StringType(), True)
            ])
        )
        # Làm sạch cột works trích xuất ra work_id
        editions_clean_work = editions.withColumn("works", from_json("works", work_schema)) \
                                    .withColumn("works", regexp_extract(col("works")[0]["key"], r"[A-Z]+\d+[A-Z]", 0))
        # Chuyển đổi kiểu dữ liệu publish_date và làm sạch cột key (cột edition_id)
        editions_clean = editions_clean_work.withColumn("publish_date", col("publish_date").cast("string")) \
                        .withColumn("key", regexp_extract("key", r"[A-Z]+\d+[A-Z]", 0))
        # Làm sạch cột publish_date
        publish_date_clean = editions_clean.withColumn("publish_date", trim(regexp_extract("publish_date", r"(\d{4})", 1))) \
                                            .withColumn("publish_date", when(col("publish_date") == "", None)
                                                                        .otherwise(col("publish_date").cast("int"))) 
        # Lấy trung vị cột publish_date
        publish_date_median = publish_date_clean.approxQuantile("publish_date", [0.5], 0.01)[0]

        # Lấy trung vị cột number_of_pages
        number_of_pages_median = publish_date_clean.approxQuantile("number_of_pages", [0.5], 0.01)[0]

        # Làm sạch giá trị null các cột
        editions_clean = publish_date_clean.fillna({"publish_date" : publish_date_median, "number_of_pages" : number_of_pages_median,
                                                    "publishers" : "unknown", "languages" : "unknown", "title" : "unknown"})
        editions_clean.select([count(when (col(c).isNull() , c)).alias(c) for c in editions.columns ]).show()
        # Làm sạch cột title
        editions_clean = editions_clean.filter(col("title").rlike("[A-Za-z]")) \
                                        .withColumnRenamed("key", "edition_id") \
                                        .withColumnRenamed("works", "work_id")
        print("Edition count : " , editions_clean.count())
        editions_clean.show(5)
        print("====== Clean editions sucessfully ======")
        return editions_clean

    def author_clean(authors) :
        print("\n" + "="*60 )
        print("====== Starting Clean authors ======")
        print("="*60 )
        # Làm sạch cột author_id
        authors_key = authors.withColumn("key", trim(regexp_replace("key", r"/authors/", ""))) \
                            .withColumnRenamed("key", "author_id")
        # Làm sạch cột name
        authors_name = authors_key.filter((col("name").isNotNull()) & (~col("name").rlike("n/a")))
        # Làm sạch cột birth_date 
        author_clean = authors_name.withColumn("birth_date", trim(regexp_extract("birth_date", r"\d{4}", 0)).cast("int")) \
                                    .withColumn("birth_date", (rand() * 200 + 1800).cast("int"))
        print("====== Clean authors sucessfully ======")
        author_clean.show(5)
        return author_clean

In [ ]:
class EtlPipeline :
    def dim_work(work_clean):
        print("\n" + "="*60 )
        print("====== Create dimension work  ======")
        print("="*60 )
        # Tạo ra bảng dim_work 
        dim_work = work_clean.select("work_id", 
                                    "title", 
                                    "first_publish_year", 
                                    "edition_count").dropDuplicates()
        dim_work.show(5)
        return dim_work
    def dim_subject(work_clean) :
        print("\n" + "="*60 )
        print("====== Create dimension subject  ======")
        print("="*60 )
        # Tạo bảng dim_subject
        dim_subject = work_clean.select("subject").dropDuplicates()
        dim_subject = dim_subject.distinct()
        dim_subject.count()
        # Gen Id cho bảng dim_subject sử dụng window function
        dim_subject = dim_subject.withColumn("subject_id", row_number().over(Window.orderBy("subject")))
        dim_subject.show(5)
        return dim_subject 

    def work_subject(work_clean, dim_subject) :
        print("\n" + "="*60 )
        print("====== Create bridghe work subject table  ======")
        print("="*60 )
        # Tạo bảng bridge work_subject
        work_subject = work_clean.join(dim_subject , on = "subject", how = "inner") \
                        .select("work_id", "subject_id") \
                        .orderBy("subject")
        work_subject.show(10)
        return work_subject
    def work_author(work_clean):
        print("\n" + "="*60 )
        print("====== Create bridghe work author table  ======")
        print("="*60 )
        # Tạo author schema
        author_schema = ArrayType(
            StructType([
                StructField("key", StringType(), True),
                StructField("name", StringType(), True)
            ])
        )
        # Trích ra cột id và name của author
        work_author = work_clean.withColumn("authors" , from_json(col("authors"), author_schema)) \
                        .withColumn("authors", explode(col("authors"))) \
                        .select("work_id","authors.key")
        # Làm sạch cột author_id
        work_author = work_author.withColumn("key", regexp_extract(col("key"), r"[A-Z]+\d+[A-Z]", 0))
        work_author = work_author.select(col("work_id"), col("key").alias("author_id"))
        work_author.show(5)
        return work_author
    ## Edition
    def dim_edition(edition_clean) :
        print("\n" + "="*60 )
        print("====== Create dimension edition ======")
        print("="*60 )
        # Tạo bảng dim_editions
        dim_edition = edition_clean.select("edition_id", "title", "publish_date", "publishers", "languages").dropDuplicates() 
        # Explode cột publishers
        dim_edition_clean = dim_edition.withColumn("publishers", from_json("publishers", ArrayType(StringType()))) \
                                        .withColumn("publishers", explode("publishers"))
        # Xóa các khoảng trắng, lọc các dữ liệu bẩn cột publishers
        dim_edition_clean = dim_edition_clean.withColumn("publishers", trim(regexp_replace("publishers", r".$|-$|\?|\]|\[|s\.n\.?|etc\.?|et al\.?|publisher not identified", ""))) \
                                            .filter( (col("publishers").rlike("[a-z]{3,}"))
                                                    & (col("publishers") != "")
                                                    & (length("publishers") < 86)
                                                    & (~col("publishers").rlike(r"printed by|printed for|distributed by|distributor|sold by")))
        # Tạo schema cho cột languages
        language_schema = ArrayType(
            StructType([
                StructField("key", StringType(), True)
            ])
        )
        # Trích xuất ra language từ cột languages
        dim_edition_clean = dim_edition_clean.withColumn("languages", from_json("languages", language_schema)) \
                                            .withColumn("languages" , regexp_extract(col("languages")[0]["key"], "[A-Z|a-z]+$", 0))
        # Đổi các giá trị null sang unknown
        dim_edition_clean = dim_edition_clean.fillna({"languages" : "unknown"})
        # Đổi tên cột publishers, languages
        dim_edition_clean = dim_edition_clean.withColumnRenamed("publishers" , "publisher") \
                                            .withColumnRenamed("languages" , "language")
        language_map = {
            "eng": "English",
            "vie": "Vietnamese",
            "fre": "French",
            "ger": "German",
            "spa": "Spanish",
            "ita": "Italian",
            "jpn": "Japanese",
            "kor": "Korean",
            "chi": "Chinese",
            "rus": "Russian",
            "por": "Portuguese",
            "ara": "Arabic",
            "hin": "Hindi",
            "ben": "Bengali",
            "tur": "Turkish",
            "pol": "Polish",
            "ukr": "Ukrainian",
            "unknown" : "Unknown"
        }
        from itertools import chain
        # Tạo mapping để map cột language sang ngôn ngữ chi tiết
        mapping = create_map(
            [lit(x) for x in chain(*language_map.items())]
        )
        # Mapping cột language
        dim_edition = dim_edition_clean.withColumn("language", mapping[col("language")])
        dim_edition.show(5)
        return dim_edition
    def dim_time(edition_clean) :
        print("\n" + "="*60 )
        print("====== Create dimension time ======")
        print("="*60 )
        dim_time = edition_clean.select(col("publish_date").alias("publish_year")).distinct()
        # Tạo ra 2 cột decade và century
        dim_time = dim_time.withColumn("decade" , (col("publish_year")/10).cast("int") * 10) \
                .withColumn("century", ((col("publish_year") - 1)/100).cast("int") + 1)
        # Gen ra time_id cho dim_time
        dim_time = dim_time.filter(col("publish_year") >= 1000) \
                            .withColumn("time_id" , row_number().over(Window.orderBy("publish_year"))) \
                            .select("time_id", "publish_year", "decade", "century")
        dim_time.show(5)
        return dim_time
    def fact_book(edition_clean, dim_time):
        print("\n" + "="*60 )
        print("====== Create fact book ======")
        print("="*60 )
        # Tạo bảng fact_book
        fact_book = edition_clean.select("edition_id", "work_id", col("publish_date").alias("publish_year"), "number_of_pages")
        # Join với bảng dim_time lấy time_id
        fact_book = fact_book.join(dim_time.select("time_id", "publish_year"), on = "publish_year", how = "left") \
                            .drop("publish_year") \
                            .select("edition_id", "work_id", "time_id", "number_of_pages")
        fact_book.show(5)
        return fact_book
    # Author
    def dim_author(author_clean) :
        print("====== Create author dimension ======")
        # Tạo bảng dim_author
        dim_author = author_clean.select("author_id", 
                                "name", 
                                "birth_date").dropDuplicates()
        dim_author.show(5)
        return dim_author

In [ ]:
class BookPipeline :
    def __init__(self):
        self.spark = SparkConfig().creat_sparksession()
        self.load = DataLoader()
        self.clean = DataClean
        self.etl = EtlPipeline
        self.result = {}
    def load_data(self) :
        print("\n" + "="*60 )
        print("====== Data Loading Phase ======")
        print("="*60 )

        self.work = self.clean(self.load.read_works())
        self.edition = self.clean(self.load.read_editions())
        self.author = self.clean(self.load.read_authors())

        print("\n All data loaded successfully")

    def run_etl_1(self) :
        try :
            result = self.etl.dim_work(self.work)
            self.result["dim_work"] = result
            return result
        except Exception as e :
            print("Error in ETL 1 : " , str(e))
            raise
    
    def run_etl_2(self) :
        try :
            result = self.etl.dim_subject(self.work)
            self.result["dim_subject"] = result
            return result
        except Exception as e :
            print("Error in ETL 2 : " , str(e))
            raise
    
    def run_etl_3(self) :
        try :
            result = self.etl.work_subject(self.work , self.result["dim_subject"])
            self.result["work_subject"] = result
            return result
        except Exception as e :
            print("Error in ETL 3 : " , str(e))
            raise
    
    def run_etl_4(self) :
        try :
            result = self.etl.work_author(self.work)
            self.result["work_author"] = result
            return result
        except Exception as e :
            print("Error in ETL 4 : " , str(e))
            raise
    
    def run_etl_5(self) :
        try :
            result = self.etl.dim_edition(self.work)
            self.result["dim_edition"] = result
            return result
        except Exception as e :
            print("Error in ETL 5 : " , str(e))
            raise
    
    def run_etl_6(self) :
        try :
            result = self.etl.dim_time(self.work)
            self.result["dim_time"] = result
            return result
        except Exception as e :
            print("Error in ETL 6 : " , str(e))
            raise
    
    def run_etl_7(self) :
        try :
            result = self.etl.dim_author(self.work)
            self.result["dim_author"] = result
            return result
        except Exception as e :
            print("Error in ETL 7 : " , str(e))
            raise
    
    def run_etl_8(self) :
        try :
            result = self.etl.fact_book(self.work, self.result["dim_time"])
            self.result["fact_book"] = result
            return result
        except Exception as e :
            print("Error in ETL 8 : " , str(e))
            raise
    
    def run_all_etl(self) :
        print("\n" + "="*60 )
        print("====== ETL PHASE - RUNNING ALL ETL ======")
        print("="*60 )

        total_start = time.time()

        self.run_etl_1()
        self.run_etl_2()
        self.run_etl_3()
        self.run_etl_4()
        self.run_etl_5()
        self.run_etl_6()
        self.run_etl_7()
        self.run_etl_8()

        total_elapsed = time.time() - total_start

        print("\n" + "="*60 )
        print(f"====== ALL ETL IN COMPLETED IN {total_elapsed:.2f}s ======")
        print("="*60 )
    
    def save_parquet(self) :
        print("\n" + "="*60 )
        print("====== STARTING SAVE TABLE ======")
        print("="*60 )

        for name, df in self.result.items() :
            df.coalesce(1).write.mode("overwrite").parquet(f"s3://mhai-bk/data_warehouse/{name}")
    
    def spark_stop(self) :
        self.spark.stop()
        print("=== Spark Session Stop ===")

In [ ]:
def main() :
    print("\n" + "="*60 )
    print("====== BOOK ANALYSIS SYSTEM ======")
    print("="*60 )

    try :
        Pipeline = BookPipeline()

        Pipeline.load_data()
        Pipeline.run_all_etl()
        Pipeline.save_parquet()

        print("\n" + "="*60 )
        print("====== ALL PIPELINE COMPLETED ======")
        print("="*60 )
    
    except Exception as e :
        print("Pipeline failed : ", str(e))
        traceback.print_exc()


In [16]:
if __name__ == "__main__" :
    main()

NameError: name 'main' is not defined